In [1]:
import json
import os
import datasets

/projects/iris/rferreir/.envs/spe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Path to the directory containing the original train.json, dev.json and test.json files
input_dir = "/projects/iris/rferreir/GeoBenchmark/data/SpartUN/SPARTUN"

# Path to the directory where to save the final train.json, dev.json and test.json files
output_dir = "/projects/iris/rferreir/GeoBenchmark/data/SpartUN"

In [3]:
def load_json(path):
    with open(path, 'r') as f:
        data = json.load(f)
    return data

## Process the data (this step will need to be done for each split)

In [49]:
splits = ['train', 'dev', 'test']
split = splits[2]

In [50]:
data_path = os.path.join(input_dir, f"{split}.json")
data = load_json(data_path)

In [51]:
for story in data['data']:
    if len(story['story']) > 1:
        print("more than 1")

In [52]:
i = 0
for story in data['data']:
    for q in story['questions']:
        i += 1
print(i)

5551


In [53]:
map_answers = {"dc":"disjoint", "ec":"touching", "po":"overlapped", "eq":"equal", "tpp":"covered by", "ntpp":"in", "tppi":"covers", "ntppi":"has"}

In [54]:
final_data = []
for story in data['data']:
    if len(story['story']) > 1:
        print("more than 1")
    for q in story['questions']:
        entry = {}
        entry["scenario_id"] = story['identifier']
        entry['question_id'] = f"{entry["scenario_id"]}-{q['q_id']}"
        entry['scenario'] = story['story'][0]
        entry['question'] = q['question']
        cand_ans = []
        for a in q['candidate_answers']:
            if a.lower() in map_answers:
                cand_ans.append(map_answers[a.lower()])
            else:
                cand_ans.append(a.lower())
        ans = []
        for a in q['answer']:
            if a.lower() in map_answers:
                ans.append(map_answers[a.lower()])
            else:
                ans.append(a.lower())
        entry['candidates_answers'] = cand_ans
        entry['answer'] = ans
        entry['type'] = q['q_type']
        entry['k_hop'] = q['question_info']['reasoning_steps']
        final_data.append(entry)

In [55]:
print(len(final_data))
print(final_data[0])

5551
{'scenario_id': '3776-0', 'question_id': '3776-0-0', 'scenario': 'Two boxes, named one and two exist in the image. Box one covers a medium yellow apple. In box two there is this box. Box two has a medium orange apple which is to the south of a medium yellow apple and touches another medium orange apple. Box two has the medium yellow apple. Medium orange apple number two is covered by this box. South of medium orange apple number one there is medium orange apple number two.', 'question': 'Is a medium yellow apple to the south of a fruit?', 'candidates_answers': ['yes', 'no'], 'answer': ['no'], 'type': 'YN', 'k_hop': 3}


In [56]:
output_path = os.path.join(output_dir, f"{split}.json")
with open(output_path, 'w') as f:
    json.dump(final_data, f)

## Converting to HF datasets

In [57]:
d1 = dataset = datasets.load_dataset('json', data_files={"test":os.path.join(output_dir,"test.json"), "train":os.path.join(output_dir,"train.json"), "validation":os.path.join(output_dir,"dev.json")})

print(d1)

# Change the path here if you want the dataset saved locally
#d1.save_to_disk(os.path.join("/projects/iris/rferreir/hub_datasets", "SpartUN"))

Generating test split: 5551 examples [00:00, 59521.43 examples/s]
Generating train split: 37095 examples [00:00, 54631.19 examples/s]
Generating validation split: 5600 examples [00:00, 60593.19 examples/s]


DatasetDict({
    test: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 5551
    })
    train: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 37095
    })
    validation: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 5600
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 5600/5600 [00:00<00:00, 219176.99 examples/s]


## Sanity Check

In [59]:
d2 = datasets.load_dataset("rfr2003/Geo_Benchmark", "SpartUN")

print(d1)
print(d2)

import hashlib
import json
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

for split in ['train', 'test', 'validation']:
    assert content_hash(d1[split]) == content_hash(d2[split])

Generating validation split: 100%|██████████| 5600/5600 [00:00<00:00, 199625.21 examples/s]


DatasetDict({
    test: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 5551
    })
    train: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 37095
    })
    validation: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 5600
    })
})
DatasetDict({
    test: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 5551
    })
    train: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'question', 'candidates_answers', 'answer', 'type', 'k_hop'],
        num_rows: 37095
    })
    validation: Dataset({
        features: ['scenario_id', 'question_id', 'scenario', 'questi